## ライブラリの読み込み

ライブラリを読み込みます．

In [6]:
import numpy as np
import statsmodels.api as sm

## データの生成

線形回帰モデルに従うデータを生成します．  
ここでは，以下の線形回帰モデルを仮定します：
$$ Y = 0.1 + \theta D + X^\top \beta + \epsilon $$
ここで，
- $\theta$が平均処置効果
- $D\in \{1, 0\}$は処置変数
- $X$は共変量
- $Y$はアウトカム
- $\epsilon$は誤差項
です．

In [14]:
rng = np.random.default_rng(0)

n = 1000

## 処置効果を3とする
treatment_effect = 3

## 共変量を生成
X_dim = 3
X = rng.normal(0,5,(n, X_dim))

## 処置を生成
xi = rng.uniform(-1,1,X_dim)
prob = 1/(1 + np.exp(-xi.dot(X.T) + rng.normal(0, 1, n)))
rand = rng.uniform(0, 1, n)
D = np.zeros((n, 1))
D[rand < prob, :] = 1

## アウトカムを生成
beta = rng.normal(0,1,X_dim)
epsilon = rng.normal(0,1,n)
Y = 0.1 + treatment_effect * D.T + X.dot(beta) + epsilon
Y = Y.T

ここでは，重回帰分析が意味を持つように，$X$と$D$は相関しているとします．  
これは$X$が交絡であることを意味します．  

In [15]:
## 相関の確認
np.corrcoef(D.T[0], X.T[0])

array([[1.        , 0.54465877],
       [0.54465877, 1.        ]])

## OLSによる平均処置効果の推定

OLSを用いて平均処置効果を推定します．  
$D$の係数が平均処置効果に対応します．以下の表では$x1$の結果がそれに対応します．

In [16]:
## 交絡をコントロールしているモデル
DX = np.concatenate([D, X], axis=1)
DX_reg = sm.add_constant(DX)
long_model = sm.OLS(Y, DX_reg)
res = long_model.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.990
Model:                            OLS   Adj. R-squared:                  0.990
Method:                 Least Squares   F-statistic:                 2.372e+04
Date:                Wed, 15 Apr 2026   Prob (F-statistic):               0.00
Time:                        03:23:28   Log-Likelihood:                -1415.1
No. Observations:                1000   AIC:                             2840.
Df Residuals:                     995   BIC:                             2865.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1483      0.053      2.807      0.0

真の値が$3$で，$x1$の推定値が$2.9149$なので，正しく推定できていることが確認できます．

## OLSによる推定（単回帰）

次に，$X$を無視してOLSを実行するとどうなるかをみてみます．

In [17]:
import statsmodels.api as sm

## 交絡をコントロールしていないモデル
D_reg = sm.add_constant(D)
long_model = sm.OLS(Y, D_reg)
res = long_model.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.425
Model:                            OLS   Adj. R-squared:                  0.425
Method:                 Least Squares   F-statistic:                     738.6
Date:                Wed, 15 Apr 2026   Prob (F-statistic):          3.54e-122
Time:                        03:24:44   Log-Likelihood:                -3422.1
No. Observations:                1000   AIC:                             6848.
Df Residuals:                     998   BIC:                             6858.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -5.0270      0.315    -15.975      0.0

このとき，真の値は$3$ですが，$x1$の推定値が$12.8344$なので，正しく推定できていないことが確認できます．

## $X$と$D$に相関がない場合

X$と$D$に相関がない場合，$X$を回帰モデルに含めて含めなくても，正しく平均処置効果を推定できます．

In [19]:
# XとDとの間に相関が無ければ、Xを無視しても良い

n = 1000

## 処置効果を5とする
treatment_effect = 3

## 共変量を生成
X_dim = 3
X = rng.normal(0,5,(n, X_dim))

## 処置を生成
rand = rng.uniform(0, 1, n)
D = np.zeros((n, 1))
D[rand < 0.7, :] = 1

## アウトカムを生成
beta = rng.normal(0,1,X_dim)
epsilon = rng.normal(0,1,n)
Y = 0.1 + treatment_effect * D.T + X.dot(beta) + epsilon
Y = Y.T

In [20]:
## 相関の確認
np.corrcoef(D.T[0], X.T[0])

array([[1.        , 0.00733306],
       [0.00733306, 1.        ]])

In [21]:
import statsmodels.api as sm

## 交絡をコントロールしているモデル
DX = np.concatenate([D, X], axis=1)
DX_reg = sm.add_constant(DX)
long_model = sm.OLS(Y, DX_reg)
res = long_model.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.965
Model:                            OLS   Adj. R-squared:                  0.965
Method:                 Least Squares   F-statistic:                     6904.
Date:                Wed, 15 Apr 2026   Prob (F-statistic):               0.00
Time:                        03:26:02   Log-Likelihood:                -1419.2
No. Observations:                1000   AIC:                             2848.
Df Residuals:                     995   BIC:                             2873.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.1151      0.059      1.945      0.0

In [22]:
import statsmodels.api as sm

## 交絡をコントロールしていないモデル
D_reg = sm.add_constant(D)
long_model = sm.OLS(Y, D_reg)
res = long_model.fit()
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.070
Model:                            OLS   Adj. R-squared:                  0.069
Method:                 Least Squares   F-statistic:                     75.44
Date:                Wed, 15 Apr 2026   Prob (F-statistic):           1.51e-17
Time:                        03:26:05   Log-Likelihood:                -3062.1
No. Observations:                1000   AIC:                             6128.
Df Residuals:                     998   BIC:                             6138.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.1718      0.305     -0.563      0.5